In [1]:
import yfinance as yf
import pandas as pd
import talib
import torch
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from torch.utils.data import TensorDataset, DataLoader
from imblearn.over_sampling import SMOTE
from sklearn.preprocessing import StandardScaler

In [2]:
# data = yf.download("^GDAXI", start="1990-01-01", end="2024-01-01")
# data.to_csv('index_stock_1990.csv')

df = pd.read_csv('index_stock_1990.csv')
print(df.shape)
print(df.head())


(8602, 6)
        Price               Close                High                 Low  \
0      Ticker              ^GDAXI              ^GDAXI              ^GDAXI   
1        Date                 NaN                 NaN                 NaN   
2  1990-01-02  1788.8900146484375  1788.8900146484375  1788.8900146484375   
3  1990-01-03     1867.2900390625     1867.2900390625     1867.2900390625   
4  1990-01-04  1830.9200439453125  1830.9200439453125  1830.9200439453125   

                 Open  Volume  
0              ^GDAXI  ^GDAXI  
1                 NaN     NaN  
2  1788.8900146484375       0  
3     1867.2900390625       0  
4  1830.9200439453125       0  


In [3]:
df = df.iloc[2:]
df = df[df['Volume'] != '0']

In [4]:
df

,Price,Close,High,Low,Open,Volume
2465,1999-11-01,5524.919921875,5549.77978515625,5496.77001953125,5518.740234375,12728800
2466,1999-11-02,5546.9501953125,5551.81982421875,5474.1201171875,5524.419921875,28336500
2467,1999-11-03,5560.8701171875,5592.08984375,5501.85986328125,5532.06005859375,40391100
2468,1999-11-04,5635.6201171875,5650.81982421875,5561.27978515625,5568.919921875,42606000
2469,1999-11-05,5658.10009765625,5680.669921875,5597.56982421875,5639.8798828125,35562700
...,...,...,...,...,...,...
8597,2023-12-21,16687.419921875,16708.349609375,16624.16015625,16667.310546875,57871300
8598,2023-12-22,16706.1796875,16735.3203125,16651.779296875,16673.30078125,46295300
8599,2023-12-27,16742.0703125,16775.7109375,16697.580078125,16727.76953125,37678900
8600,2023-12-28,16701.55078125,16783.7890625,16688.51953125,16780.94921875,36091600


In [5]:
df['Doji'] = talib.CDLDOJI(df['Open'], df['High'], df['Low'], df['Close'])
df['Hammer'] = talib.CDLHAMMER(df['Open'], df['High'], df['Low'], df['Close'])
df['Engulfing'] = talib.CDLENGULFING(df['Open'], df['High'], df['Low'], df['Close'])


In [6]:
df = df.rename(columns = {'Price':'Date'})

In [7]:
df = df.reset_index()

In [8]:
df.drop(columns='index', inplace=True)
df.head(20)

,Date,Close,High,Low,Open,Volume,Doji,Hammer,Engulfing
0,1999-11-01,5524.919921875,5549.77978515625,5496.77001953125,5518.740234375,12728800,0,0,0
1,1999-11-02,5546.9501953125,5551.81982421875,5474.1201171875,5524.419921875,28336500,0,0,0
2,1999-11-03,5560.8701171875,5592.08984375,5501.85986328125,5532.06005859375,40391100,0,0,0
3,1999-11-04,5635.6201171875,5650.81982421875,5561.27978515625,5568.919921875,42606000,0,0,0
4,1999-11-05,5658.10009765625,5680.669921875,5597.56982421875,5639.8798828125,35562700,0,0,0
5,1999-11-08,5647.93994140625,5681.02001953125,5613.9501953125,5628.740234375,23301600,0,0,0
6,1999-11-09,5694.72998046875,5755.25,5654.330078125,5654.330078125,32825400,0,0,0
7,1999-11-10,5742.419921875,5742.419921875,5661.68017578125,5685.5,32550800,0,0,0
8,1999-11-11,5802.35986328125,5803.83984375,5714.47998046875,5714.47998046875,36837500,0,0,0
9,1999-11-12,5791.0498046875,5824.2998046875,5750.64990234375,5789.5,35220000,0,0,0


In [9]:
df.value_counts(['Doji', 'Hammer', 'Engulfing'])

Doji  Hammer  Engulfing
0     0        0           4719
100   0        0            821
0     0       -100          234
               100          193
      100      0            131
100   100      0             20
0     100     -100            5
               100            2
100   0        100            2
0     0        80             1
Name: count, dtype: int64

In [10]:
def define_pattern(x):
    # No pattern
    if x['Doji'] == 0 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 0
    
    # Doji
    elif x['Doji'] == 100 and x['Hammer'] == 0 and x['Engulfing'] == 0:
        return 1
    
    # Hammer
    elif x['Doji'] == 0 and x['Hammer'] == 100 and x['Engulfing'] == 0:
        return 2
    
    # Engulfing
    elif x['Doji'] == 0 and x['Hammer'] == 0 and (x['Engulfing'] != 0):
        return 3
    
    # 2 classes at once
    else:
        return -1 
    


In [11]:
df['Pattern'] = df.apply(define_pattern, axis=1)

In [12]:
df

,Date,Close,High,Low,Open,Volume,Doji,Hammer,Engulfing,Pattern
0,1999-11-01,5524.919921875,5549.77978515625,5496.77001953125,5518.740234375,12728800,0,0,0,0
1,1999-11-02,5546.9501953125,5551.81982421875,5474.1201171875,5524.419921875,28336500,0,0,0,0
2,1999-11-03,5560.8701171875,5592.08984375,5501.85986328125,5532.06005859375,40391100,0,0,0,0
3,1999-11-04,5635.6201171875,5650.81982421875,5561.27978515625,5568.919921875,42606000,0,0,0,0
4,1999-11-05,5658.10009765625,5680.669921875,5597.56982421875,5639.8798828125,35562700,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...
6123,2023-12-21,16687.419921875,16708.349609375,16624.16015625,16667.310546875,57871300,0,0,0,0
6124,2023-12-22,16706.1796875,16735.3203125,16651.779296875,16673.30078125,46295300,0,0,0,0
6125,2023-12-27,16742.0703125,16775.7109375,16697.580078125,16727.76953125,37678900,0,0,0,0
6126,2023-12-28,16701.55078125,16783.7890625,16688.51953125,16780.94921875,36091600,0,0,-100,3


In [13]:
df['Pattern'].value_counts()

Pattern
 0    4719
 1     821
 3     428
 2     131
-1      29
Name: count, dtype: int64

In [14]:
df = df[df['Pattern'] != -1]
df = df.drop(columns =['Doji', 'Hammer', 'Engulfing'])

In [15]:
df.head(20)

,Date,Close,High,Low,Open,Volume,Pattern
0,1999-11-01,5524.919921875,5549.77978515625,5496.77001953125,5518.740234375,12728800,0
1,1999-11-02,5546.9501953125,5551.81982421875,5474.1201171875,5524.419921875,28336500,0
2,1999-11-03,5560.8701171875,5592.08984375,5501.85986328125,5532.06005859375,40391100,0
3,1999-11-04,5635.6201171875,5650.81982421875,5561.27978515625,5568.919921875,42606000,0
4,1999-11-05,5658.10009765625,5680.669921875,5597.56982421875,5639.8798828125,35562700,0
5,1999-11-08,5647.93994140625,5681.02001953125,5613.9501953125,5628.740234375,23301600,0
6,1999-11-09,5694.72998046875,5755.25,5654.330078125,5654.330078125,32825400,0
7,1999-11-10,5742.419921875,5742.419921875,5661.68017578125,5685.5,32550800,0
8,1999-11-11,5802.35986328125,5803.83984375,5714.47998046875,5714.47998046875,36837500,0
9,1999-11-12,5791.0498046875,5824.2998046875,5750.64990234375,5789.5,35220000,0


## Building neural network with Torch

In [16]:
# Kaiming initialization for better gradient flow through ReLU
W1 = torch.nn.init.kaiming_uniform_(torch.empty(5, 8), nonlinearity='relu').requires_grad_(True)
b1 = torch.zeros(8, requires_grad=True)
W2 = torch.nn.init.kaiming_uniform_(torch.empty(8, 4), nonlinearity='relu').requires_grad_(True)
b2 = torch.zeros(4, requires_grad=True)

In [17]:
def forward_pass(X):
    hidden_lay_pre_act = torch.matmul(X, W1) + b1
    relu_act = torch.relu(hidden_lay_pre_act)
    logits = torch.matmul(relu_act, W2) + b2
    return logits

In [18]:
freq = df['Pattern'].value_counts().sort_index()
weights = 1 / (freq ** 0.5)
weights = weights / weights.sum()  # normalize so scale doesn't affect loss magnitude

In [19]:
tensor_1 = torch.tensor(weights.values, dtype=torch.float32)
criterion = torch.nn.CrossEntropyLoss(weight = tensor_1)

In [20]:
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)

In [21]:
for epoch in range(100):
    logits = forward_pass(torch.tensor(df[['Close', 'High', 'Low', 'Open', 'Volume']].astype('float32').values, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(df['Pattern'].values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if epoch % 10 == 0:
        print(loss.item())

71085344.0
19559000.0
13491074.0
7542323.0
3331416.0
571146.9375
1933899.625
988999.0625
255015.265625
606298.5


## Doing using normal train-test split

In [22]:
# df = df[df['Pattern'] != 3]

In [23]:
df_train, df_test = train_test_split(df, test_size = 0.2)

In [24]:
X_train = df_train[['Close', 'High', 'Low', 'Open', 'Volume']]
X_test = df_test[['Close', 'High', 'Low', 'Open', 'Volume']]
y_train = df_train['Pattern']
y_test = df_test['Pattern']


In [25]:
scaler_normal = StandardScaler()
X_train_scaled = scaler_normal.fit_transform(X_train)
X_test_scaled = scaler_normal.transform(X_test)

In [32]:
# Reset optimizer so Adam momentum doesn't carry over from the sanity-check loop above
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)

for epoch in range(500):
    logits = forward_pass(torch.tensor(X_train_scaled, dtype=torch.float32))
    loss = criterion(logits, torch.tensor(y_train.values, dtype=torch.long))
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

In [33]:
logits_test = forward_pass(torch.tensor(X_test_scaled, dtype=torch.float32))
logits_test

tensor([[ 0.5600, -0.7255, -1.8128, -0.6321],
        [ 0.7795,  0.0884, -2.1157, -0.9923],
        [ 1.0625,  0.5678, -0.6503, -0.3584],
        ...,
        [ 0.5555, -0.4387, -0.6848, -0.7526],
        [ 0.5787, -0.2428, -1.7099, -0.9440],
        [ 2.3097,  1.2824,  1.1793,  1.5468]], grad_fn=<AddBackward0>)

In [47]:
test_result = torch.argmax(logits_test, dim=1)
test_result

tensor([0, 0, 0,  ..., 0, 0, 0])

In [48]:
print(classification_report(y_test, test_result.numpy()))

              precision    recall  f1-score   support

           0       0.78      1.00      0.88       954
           1       0.00      0.00      0.00       155
           2       0.00      0.00      0.00        26
           3       0.00      0.00      0.00        85

    accuracy                           0.78      1220
   macro avg       0.20      0.25      0.22      1220
weighted avg       0.61      0.78      0.69      1220



In [30]:
# Use scaled data (same scaler fitted on X_train above)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

dataset = TensorDataset(X_train_tensor, y_train_tensor)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [31]:
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)  # reset before new experiment

for epoch in range(500):
    for X_batch, y_batch in loader:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

1.3683979511260986
1.2354280948638916
1.3413268327713013
1.187791347503662
1.4376529455184937
1.2756632566452026
1.5361380577087402
1.3462456464767456
1.5045526027679443
1.3405340909957886
1.4318156242370605
1.5218842029571533
1.264201283454895
1.6812466382980347
1.4126776456832886
1.3182190656661987
1.2217377424240112
1.598706603050232
1.1765296459197998
1.2182198762893677
1.2267177104949951
1.1418014764785767
1.1102255582809448
1.308485746383667
1.1208410263061523
1.1444456577301025
2.3586888313293457
1.9335418939590454
1.1118903160095215
1.2206823825836182
1.2584141492843628
1.2579829692840576
1.4994057416915894
1.2152514457702637
1.3778671026229858
1.512357473373413
1.0858149528503418
1.2391334772109985
1.187421202659607
1.1717884540557861
1.2209758758544922
1.417731523513794
1.435203194618225
1.124928593635559
1.1459952592849731
1.2412678003311157
1.3177968263626099
1.5665514469146729
1.4605919122695923
1.281632661819458
1.3232190608978271
1.2374179363250732
1.512174367904663
1.31

In [27]:
logits_test_batch = forward_pass(torch.tensor(X_test_scaled, dtype=torch.float32))
test_result = torch.argmax(logits_test_batch, dim=1)
print(classification_report(y_test, test_result.numpy()))

              precision    recall  f1-score   support

           0       0.78      1.00      0.88       954
           1       0.00      0.00      0.00       155
           2       0.00      0.00      0.00        26
           3       0.00      0.00      0.00        85

    accuracy                           0.78      1220
   macro avg       0.20      0.25      0.22      1220
weighted avg       0.61      0.78      0.69      1220



C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [28]:
y_train.value_counts()

Pattern
0    3765
1     666
3     343
2     105
Name: count, dtype: int64

In [30]:
sm = SMOTE(sampling_strategy={1:700, 2:400, 3:400})
X_train_partial_sm, y_train_partial_sm = sm.fit_resample(X_train_scaled, y_train)

## Applying imbalanced learn

In [36]:
sm = SMOTE()
X_train_sm, y_train_sm = sm.fit_resample(X_train.astype('float32').values, y_train)

In [37]:
y_train_sm.value_counts()

Pattern
0    3765
1    3765
3    3765
2    3765
Name: count, dtype: int64

In [38]:
scaler = StandardScaler()
scaler_1 = scaler.fit(X_train)
X_train_sm = scaler_1.transform(X_train_sm)
X_test_sm = scaler_1.transform(X_test)

C:\Users\USER\candle_classifier\candle_venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


In [39]:
X_train_sm

array([[-0.58390933, -0.55967516, -0.5698433 , -0.54624164,  1.2619584 ],
       [-0.47139928, -0.4570556 , -0.46555275, -0.44230607, -1.2603146 ],
       [ 1.1155282 ,  1.1021467 ,  1.1271442 ,  1.1084268 , -0.54661024],
       ...,
       [-1.3196598 , -1.3102406 , -1.3101256 , -1.3043088 ,  0.02729392],
       [-0.21962117, -0.2208672 , -0.22492434, -0.21510752,  1.3451775 ],
       [-0.28852475, -0.29957342, -0.29604208, -0.3083971 ,  1.3411896 ]],
      shape=(15060, 5), dtype=float32)

In [40]:
X_train_sm = torch.tensor(X_train_sm, dtype=torch.float32)
y_train_sm = torch.tensor(y_train_sm, dtype=torch.long)

dataset = TensorDataset(X_train_sm, y_train_sm)
loader = DataLoader(dataset, batch_size=32, shuffle=True)

In [41]:
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)  # reset before new experiment

for epoch in range(500):
    for X_batch, y_batch in loader:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 250 == 0:
            print(loss.item())

1.8146048784255981
1.6757844686508179
1.346396565437317
1.9364004135131836
1.262266993522644
1.3804328441619873
1.3536791801452637
1.188428282737732
1.2854279279708862
1.3124098777770996
1.2788856029510498
1.3004262447357178
1.016014575958252
1.446725606918335
1.0513967275619507
1.3632997274398804
1.305490493774414
1.2900269031524658
1.2726422548294067
1.602434515953064
1.0171245336532593
1.140562891960144
1.4295271635055542
1.014046549797058
1.2152119874954224
1.1736913919448853
1.1325024366378784
1.2042680978775024
1.173904299736023
1.0388586521148682
1.1225509643554688
1.1786727905273438
0.9966134428977966
1.29963219165802
1.2165166139602661
1.1952911615371704
1.1982295513153076
1.2399109601974487
1.0242294073104858
1.0668858289718628
1.0222373008728027
1.1626145839691162
1.0398261547088623
1.1967129707336426
1.133943796157837
1.3169987201690674
1.1396335363388062
1.209913730621338
1.2273669242858887
1.201558232307434
1.0201565027236938
1.1625436544418335
1.2053723335266113
1.096151

0.5172162652015686
0.7675524353981018
0.46126100420951843
0.7540061473846436
0.7144928574562073
0.5384278297424316
0.5280279517173767
0.8037903904914856
0.868818461894989
0.6867082715034485
0.5847914218902588
0.629662036895752
0.4856443703174591
0.5373178124427795
0.7218823432922363
0.6268843412399292
0.670333981513977
0.8968107104301453
0.5287569165229797
0.4719873070716858
0.6378461718559265
0.7361277341842651
0.820452868938446
0.6185222864151001
0.6165167093276978
0.6351497173309326
0.7018726468086243
0.8749021887779236
0.7762973308563232
0.49969223141670227
0.5193838477134705
0.8236926794052124
0.6035158634185791
0.4918617904186249
0.4970048666000366
1.0415408611297607
0.5629149675369263
0.5388562083244324
0.40120890736579895
0.5136284232139587
0.7380257844924927
0.644822895526886
0.5896650552749634
0.5274903178215027
0.6255506277084351
0.5522310137748718
0.6478070616722107
0.6246514916419983
0.7619503736495972
1.077733039855957
0.7862426042556763
0.4410301744937897
0.6384210586547

In [42]:
logits_test_batch_sm = forward_pass(torch.tensor(X_test_sm,dtype=torch.float32))
test_result_sm = torch.argmax(logits_test_batch_sm, dim=1)
print(classification_report(y_test, test_result_sm.numpy()))

              precision    recall  f1-score   support

           0       1.00      0.03      0.05       954
           1       0.30      0.70      0.42       155
           2       0.06      0.73      0.11        26
           3       0.10      0.64      0.18        85

    accuracy                           0.17      1220
   macro avg       0.37      0.52      0.19      1220
weighted avg       0.83      0.17      0.11      1220



## Using partial smote for class 1,2,3

In [43]:
sm = SMOTE(sampling_strategy={2:400, 3:400})
X_train_partial_sm, y_train_partial_sm= sm.fit_resample(X_train_scaled, y_train)

In [44]:
X_train_partial_sm_tensor = torch.tensor(X_train_partial_sm, dtype=torch.float32)
y_train_partial_sm_tensor = torch.tensor(y_train_partial_sm, dtype=torch.long)

dataset_partial_sm = TensorDataset(X_train_partial_sm_tensor, y_train_partial_sm_tensor)
loader_partial_sm = DataLoader(dataset_partial_sm, batch_size=32, shuffle=True)

In [45]:
optimizer = torch.optim.Adam([W1, b1, W2, b2], lr=0.01)  # reset before new experiment

for epoch in range(200):
    for X_batch, y_batch in loader_partial_sm:
        logits = forward_pass(X_batch)
        loss = criterion(logits, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        if epoch % 50 == 0:
            print(loss.item())

2.581422805786133
1.344353437423706
1.2909111976623535
1.038718342781067
0.8035333752632141
1.2843583822250366
1.148677110671997
1.0426347255706787
0.8620791435241699
0.8318274021148682
0.7939789891242981
0.7621610164642334
1.107888102531433
0.9492260217666626
0.8862889409065247
0.867136538028717
1.1376426219940186
1.321047067642212
0.8667741417884827
0.6180953979492188
0.7687264680862427
0.8547194600105286
0.8550924062728882
1.0116513967514038
1.1438443660736084
0.7529032826423645
0.7245752811431885
1.068230390548706
1.0046495199203491
1.2739979028701782
0.8841224312782288
1.035673975944519
0.9055673480033875
0.7666152119636536
1.2680282592773438
0.7417288422584534
1.2144334316253662
0.6933268308639526
1.5763566493988037
1.2353918552398682
0.7838030457496643
1.102563500404358
1.0454312562942505
0.6801433563232422
1.1102544069290161
1.0196419954299927
1.0900884866714478
0.7472635507583618
0.4997062683105469
0.7335385680198669
1.1122727394104004
1.2689827680587769
1.111595630645752
1.33

0.5886653065681458
0.8049275875091553
0.6936266422271729
0.7185914516448975
1.2097969055175781
0.8804462552070618
0.6791892051696777
1.2610100507736206
0.6834152936935425
0.9618784785270691
1.3300186395645142
1.2394710779190063
0.7520304322242737
0.910289466381073
0.8665129542350769
0.7554741501808167
0.962151050567627
0.6284769773483276
0.8141223192214966
1.2515394687652588
0.5932836532592773
0.48572587966918945
0.9150794148445129
0.7652091979980469
0.8535104393959045
1.1649017333984375
1.1014275550842285
0.9562067985534668
0.8549118638038635
1.1043568849563599
1.0507093667984009
0.9440914988517761
1.0060172080993652
0.9070354700088501
0.7134177684783936
0.7937077879905701
0.7930923700332642
0.6765305995941162
0.6495836973190308
0.6456162333488464
0.9462367296218872
0.5241918563842773
0.9404336214065552
0.755715012550354
1.1401976346969604
1.3261722326278687
0.9309921264648438
1.1693346500396729
0.8968654870986938
0.7755945920944214
0.8809536099433899
0.7224814295768738
1.015693902969

In [46]:
logits_test_batch_partial_sm = forward_pass(torch.tensor(X_test_scaled,dtype=torch.float32))
test_result_partial_sm = torch.argmax(logits_test_batch_partial_sm, dim=1)
print(classification_report(y_test, test_result_partial_sm.numpy()))

              precision    recall  f1-score   support

           0       0.86      0.77      0.81       954
           1       0.47      0.58      0.52       155
           2       0.15      0.85      0.26        26
           3       0.19      0.07      0.10        85

    accuracy                           0.70      1220
   macro avg       0.42      0.57      0.42      1220
weighted avg       0.75      0.70      0.71      1220



There are too little data for 2 and 3 in reality, so SMOTE for new data is not possible. Tried and harmed the model. 
- Deletion is also not possible, because it would turn into binary classification which kind of useless

## Save the trained weights

In [49]:
torch.save({
    'W1': W1.detach(),
    'b1': b1.detach(),
    'W2': W2.detach(),
    'b2': b2.detach(),
    'scaler': scaler_normal  # save the scaler too!
}, 'candle_model.pt')